# 02 — Experiment Analysis

FDR-corrected significance tests mirroring the source case study's 4 Key Insights, an
interaction-regression cross-check, and a power analysis explaining why
nothing survives correction at the current sample size. See
`experiment_analysis.py` for full docstrings.

In [1]:
import sys

sys.path.insert(0, "..")

import pandas as pd

from experiment_analysis import (
    run_named_comparisons,
    fit_interaction_models,
    required_sample_size,
    NAMED_COMPARISONS,
    TIERS_ASC,
)

experiment_log = pd.read_csv("../data/processed/experiment_log.csv")

## Named comparisons (mirrors the source case study's 4 Key Insights), FDR-corrected

In [2]:
comparisons = run_named_comparisons(experiment_log)
comparisons.to_csv("../outputs/named_comparisons.csv", index=False)
comparisons[["comparison", "tier", "metric", "pct_lift", "p_value", "q_value", "significant"]]

,comparison,tier,metric,pct_lift,p_value,q_value,significant
0,voucher_vs_none,0-29,order_rate,3.49602,0.42894,0.98064,False
1,voucher_vs_none,0-29,profit_per_user,5.10436,0.28278,0.56557,False
2,voucher_vs_none,30-69,order_rate,-5.34415,0.34903,0.98064,False
3,voucher_vs_none,30-69,profit_per_user,-15.16982,0.00898,0.17560,False
4,voucher_vs_none,70-79,order_rate,27.07146,0.00807,0.19365,False
5,voucher_vs_none,70-79,profit_per_user,4.16459,0.67108,0.89478,False
6,voucher_vs_none,80-89,order_rate,16.50106,0.09832,0.58994,False
7,voucher_vs_none,80-89,profit_per_user,-1.25217,0.89815,0.97250,False
8,voucher_vs_none,90-99,order_rate,29.59787,0.04117,0.32933,False
9,voucher_vs_none,90-99,profit_per_user,19.77420,0.18298,0.54987,False


**Headline finding: 0 of 48 comparisons are significant after FDR
correction**, despite most trending in the direction the source case study reports.
This is the most important result in this notebook, not a footnote —
see the power analysis below for why.

## Interaction regression (tier x condition), cross-check

In [3]:
order_summary, profit_summary = fit_interaction_models(experiment_log)
print("Order (logit) — top 10 terms by |coef|:")
order_summary.reindex(order_summary["coef"].abs().sort_values(ascending=False).index).head(10)

Order (logit) — top 10 terms by |coef|:


,term,coef,std_err,p_value
0,Intercept,-2.43605,0.03385,0.00000
5,C(tier)[T.100],-0.93859,0.08171,0.00000
38,C(tier)[T.70-79]:C(condition)[T.s0_m199_c1],0.41548,0.10921,0.00014
35,C(tier)[T.90-99]:C(condition)[T.s0_m299_c3],0.37999,0.14538,0.00895
13,C(tier)[T.70-79]:C(condition)[T.s19_m199_c3],0.35387,0.10962,0.00125
28,C(tier)[T.70-79]:C(condition)[T.s19_m199_c1],0.31018,0.10991,0.00477
1,C(tier)[T.30-69],0.26533,0.05696,0.00000
20,C(tier)[T.90-99]:C(condition)[T.s0_m249_c3],0.25191,0.14974,0.09249
23,C(tier)[T.70-79]:C(condition)[T.s9_m249_c3],0.23657,0.11034,0.03203
18,C(tier)[T.70-79]:C(condition)[T.s0_m249_c3],0.22559,0.11018,0.04061


In [4]:
print("Profit (OLS) — top 10 terms by |coef|:")
profit_summary.reindex(profit_summary["coef"].abs().sort_values(ascending=False).index).head(10)

Profit (OLS) — top 10 terms by |coef|:


,term,coef,std_err,p_value
0,Intercept,0.15385,0.00407,0.00000
5,C(tier)[T.100],-0.12250,0.00715,0.00000
4,C(tier)[T.90-99],-0.09992,0.01399,0.00000
3,C(tier)[T.80-89],-0.07693,0.01014,0.00000
2,C(tier)[T.70-79],-0.06146,0.00965,0.00000
17,C(tier)[T.30-69]:C(condition)[T.s0_m249_c3],-0.03288,0.01035,0.00149
35,C(tier)[T.90-99]:C(condition)[T.s0_m299_c3],0.02943,0.01959,0.13300
22,C(tier)[T.30-69]:C(condition)[T.s9_m249_c3],-0.02567,0.01036,0.01322
13,C(tier)[T.70-79]:C(condition)[T.s19_m199_c3],0.02417,0.01368,0.07737
28,C(tier)[T.70-79]:C(condition)[T.s19_m199_c1],0.02265,0.01375,0.09951


## Sample size needed: worked example (90-99% tier, voucher_vs_none)

In [5]:
n_comparisons = len(NAMED_COMPARISONS) * len(TIERS_ASC)
example = comparisons[
    (comparisons["comparison"] == "voucher_vs_none")
    & (comparisons["tier"] == "90-99")
    & (comparisons["metric"] == "order_rate")
].iloc[0]
n_needed = required_sample_size(
    baseline_rate=example["mean_b"], pct_lift=example["pct_lift"], n_comparisons=n_comparisons
)
current_n = experiment_log[
    (experiment_log["tier"] == "90-99") & (experiment_log["condition"] == "no_voucher")
].shape[0]

print(f"Observed lift: {example['pct_lift']:.1f}% on a {example['mean_b']:.3f} baseline order rate")
print(f"Required N per arm (80% power, Bonferroni-corrected for {n_comparisons} tests): {n_needed:,}")
print(f"Current simulated N in this cell: {current_n:,}")
print("UNDERPOWERED" if current_n < n_needed else "Adequately powered")

Observed lift: 29.6% on a 0.091 baseline order rate
Required N per arm (80% power, Bonferroni-corrected for 24 tests): 4,000
Current simulated N in this cell: 1,088
UNDERPOWERED
